## Notebook 05 — Missing Value Fraud Rate Analysis

---

This notebook:
1. Loads + merges data, drops >80% missing columns, removes `TransactionID`
2. Performs stratified 80/20 train-validation split
3. For each column in X_train, computes:
   - Missing % in train
   - Fraud rate when value IS missing
   - Fraud rate when value is NOT missing
   - Absolute difference between the two rates
4. Prints top 20 columns sorted by absolute difference (descending)

> **No data modification or preprocessing is performed.**

---
## 📦 Step 0 — Import Libraries, Load & Split Data

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'
TARGET    = 'isFraud'

# Load & merge
print('Loading and merging...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Stratified 80/20 split
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
del train, X, y

print(f'✅ X_train shape : {X_train.shape}')
print(f'✅ X_val shape   : {X_val.shape}')
print(f'✅ Train fraud % : {y_train.mean()*100:.2f}%')
print(f'✅ Val fraud %   : {y_val.mean()*100:.2f}%')

Loading and merging...
✅ X_train shape : (472432, 358)
✅ X_val shape   : (118108, 358)
✅ Train fraud % : 3.50%
✅ Val fraud %   : 3.50%


---
## 🔍 Step 1 — Compute Fraud Rate by Missingness for Each Column

In [12]:
# Combine X_train and y_train for analysis
train_df = X_train.copy()
train_df[TARGET] = y_train.values

records = []
for col in X_train.columns:
    is_missing = train_df[col].isnull()
    missing_count = is_missing.sum()
    if missing_count == 0:
        continue  # skip fully complete columns

    missing_pct_col  = missing_count / len(train_df) * 100
    fraud_when_miss  = train_df.loc[is_missing,  TARGET].mean() * 100
    fraud_when_present = train_df.loc[~is_missing, TARGET].mean() * 100
    abs_diff         = abs(fraud_when_miss - fraud_when_present)

    records.append({
        'Column'             : col,
        'Missing %'         : round(missing_pct_col, 2),
        'Fraud % (missing)' : round(fraud_when_miss, 2),
        'Fraud % (present)' : round(fraud_when_present, 2),
        'Abs Difference'    : round(abs_diff, 2)
    })

fraud_miss_df = pd.DataFrame(records).sort_values('Abs Difference', ascending=False).reset_index(drop=True)
fraud_miss_df.index += 1

print(f'✅ Analysed {len(fraud_miss_df)} columns with at least 1 missing value.')

✅ Analysed 340 columns with at least 1 missing value.


---
## 📊 Step 2 — Top 20 Features by Fraud Rate Difference

In [13]:
top20 = fraud_miss_df.head(20)

print('=' * 80)
print('   TOP 20 FEATURES — Fraud Rate Difference (Missing vs Present)        ')
print('=' * 80)
print(f'{"Rank":<5} {"Column":<20} {"Missing %":>10} {"Fraud%(miss)":>13} {"Fraud%(pres)":>13} {"Abs Diff":>10}')
print('-' * 80)
for rank, row in top20.iterrows():
    print(
        f'{rank:<5} {row["Column"]:<20} '
        f'{row["Missing %"]:>9.2f}% '
        f'{row["Fraud % (missing)"]:>12.2f}% '
        f'{row["Fraud % (present)"]:>12.2f}% '
        f'{row["Abs Difference"]:>9.2f}%'
    )
print('=' * 80)

   TOP 20 FEATURES — Fraud Rate Difference (Missing vs Present)        
Rank  Column                Missing %  Fraud%(miss)  Fraud%(pres)   Abs Diff
--------------------------------------------------------------------------------
1     addr1                    11.16%        11.76%         2.46%      9.30%
2     addr2                    11.16%        11.76%         2.46%      9.30%
3     V291                      0.00%        12.50%         3.50%      9.00%
4     V298                      0.00%        12.50%         3.50%      9.00%
5     V311                      0.00%        12.50%         3.50%      9.00%
6     V310                      0.00%        12.50%         3.50%      9.00%
7     V309                      0.00%        12.50%         3.50%      9.00%
8     V308                      0.00%        12.50%         3.50%      9.00%
9     V307                      0.00%        12.50%         3.50%      9.00%
10    V306                      0.00%        12.50%         3.50%      9.00%


---
## 🔄 Step 3 — Refined Analysis (missing_count >= 500, full precision)

In [14]:
# Refined: only columns where missing_count >= 500
train_df = X_train.copy()
train_df[TARGET] = y_train.values

records = []
for col in X_train.columns:
    is_missing    = train_df[col].isnull()
    missing_count = int(is_missing.sum())
    if missing_count < 500:
        continue
    missing_pct_col    = missing_count / len(train_df) * 100
    fraud_when_miss    = train_df.loc[is_missing,  TARGET].mean() * 100
    fraud_when_present = train_df.loc[~is_missing, TARGET].mean() * 100
    abs_diff           = abs(fraud_when_miss - fraud_when_present)
    records.append({
        'Column'            : col,
        'Missing Count'     : missing_count,
        'Missing %'         : missing_pct_col,
        'Fraud % (missing)' : fraud_when_miss,
        'Fraud % (present)' : fraud_when_present,
        'Abs Difference'    : abs_diff
    })

fraud_df = (pd.DataFrame(records)
              .sort_values('Abs Difference', ascending=False)
              .reset_index(drop=True))
fraud_df.index += 1
top20 = fraud_df.head(20)

W    = 105
sep  = '=' * W
dash = '-' * W
hdr  = (f'{"Rank":<5} {"Column":<20} {"Miss Count":>12} '
        f'{"Missing %":>15} {"Fraud%(miss)":>16} {"Fraud%(pres)":>16} {"Abs Diff":>15}')

print(f'Refined analysis: {len(fraud_df)} columns have missing_count >= 500')
print()
print(sep)
print('   TOP 20 — Fraud Rate Difference | missing_count >= 500 | sorted by Abs Diff   ')
print(sep)
print(hdr)
print(dash)
for rank, row in top20.iterrows():
    print(
        f'{rank:<5} {row["Column"]:<20} {row["Missing Count"]:>12,} '
        f'{row["Missing %"]:>14.6f}% '
        f'{row["Fraud % (missing)"]:>15.6f}% '
        f'{row["Fraud % (present)"]:>15.6f}% '
        f'{row["Abs Difference"]:>14.6f}%'
    )
print(sep)

Refined analysis: 265 columns have missing_count >= 500

   TOP 20 — Fraud Rate Difference | missing_count >= 500 | sorted by Abs Diff   
Rank  Column                 Miss Count       Missing %     Fraud%(miss)     Fraud%(pres)        Abs Diff
---------------------------------------------------------------------------------------------------------
1     addr1                      52,723      11.159913%       11.763367%        2.460753%       9.302614%
2     addr2                      52,723      11.159913%       11.763367%        2.460753%       9.302614%
3     id_13                     370,347      78.391599%        2.176877%        8.295048%       6.118171%
4     R_emaildomain             362,236      76.674738%        2.087037%        8.140041%       6.053004%
5     id_06                     362,641      76.760465%        2.142615%        7.978796%       5.836181%
6     id_05                     362,641      76.760465%        2.142615%        7.978796%       5.836181%
7     id_02   